# HDAR Host B — Independent Execution on Google Colab

This notebook demonstrates **cross-platform agent continuation**:
- **Host A**: macOS (where the capsule was signed)
- **Host B**: Google Colab Linux (this notebook)
- **Verifier**: Run on a third machine or locally

The capsule was created and owner-signed on macOS. Colab restores it on Linux,
verifies the owner signature, continues the task, and produces a successor capsule.

**This is a genuinely independent host** — different OS, different hardware, different network, different administrative domain.

## Step 1: Install dependencies

In [ ]:
!pip install cryptography -q

## Step 2: Set up working directory

In [ ]:
import os, json, hashlib, platform, socket

WORKDIR = '/content/hdar-host-b'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')
print(f'Platform: {platform.platform()}')
print(f'Hostname: {socket.gethostname()}')

## Step 3: Upload deploy package files

Upload these files to Colab (sidebar -> Files -> Upload):
1. `run_on_host_b.py`
2. `transport_capsule_epoch_1_signed.tar.gz`
3. `third_party_verifier.py` (optional)

Or download from your tunnel URL if active.

In [ ]:
# Check if files exist
has_runner = os.path.exists('run_on_host_b.py')
has_capsule = os.path.exists('transport_capsule_epoch_1_signed.tar.gz')

print(f'run_on_host_b.py: {"FOUND" if has_runner else "MISSING"}')
print(f'transport_capsule_epoch_1_signed.tar.gz: {"FOUND" if has_capsule else "MISSING"}')

if not has_runner or not has_capsule:
    print()
    print('Upload these files to Colab (sidebar -> Files -> Upload):')
    print('  1. run_on_host_b.py')
    print('  2. transport_capsule_epoch_1_signed.tar.gz')
    print()
    print('Or download from your tunnel URL:')
    print('  !curl -fsS https://YOUR-TUNNEL.loca.lt/run_on_host_b.py -o run_on_host_b.py')
    print('  !curl -fsS https://YOUR-TUNNEL.loca.lt/transport_capsule_epoch_1_signed.tar.gz -o transport_capsule_epoch_1_signed.tar.gz')

## Step 4: Write Host A build report

In [ ]:
host_a_report = {
  "capsule_epoch_1": {
    "agent_id": "hdar-seed-poc-agent",
    "epoch": 1,
    "file_count": 5,
    "manifest_hash": "228dff4fb4d59b271caf31c265e308bd5fc1017cc9498b4fd74fd310fd62ac6c",
    "ok": True,
    "owner_public_key": "8bb30fed7ef7202b6ac738ad6c4680fe2841976eecfbcbbf42462d4a4448d80d",
    "owner_signature_algorithm": "ed25519",
    "owner_signed": True,
    "problems": [],
    "signature_mode": "ed25519-owner-signed",
    "total_size": 11719,
    "workspace_root_hash": "45d811a423318d5ca553c4d27c88916e8ef25fd4017771ffe99d8bf6a1a2bbfa"
  },
  "claim_boundary": "Owner-signed portable capsule with Ed25519 owner authorization, multi-stage analysis pipeline, Host B signature verification, evidence packet signing, and third-party verifier support. Ready for external Host B reproduction.",
  "host_a_platform": "macOS-26.5.2-arm64-arm-64bit-Mach-O",
  "host_a_runtime_destroyed": True,
  "host_a_workspace_hash_before_destroy": "45d811a423318d5ca553c4d27c88916e8ef25fd4017771ffe99d8bf6a1a2bbfa",
  "next_real_seed_step": "Run on independent Host B with --owner-public-key and --host-a-report flags, then run third_party_verifier.py",
  "schema": "hdar.second-host-demo-build/v0.2",
  "transport_bundle": {
    "bytes": 41998,
    "path": "run_on_host_b.py",
    "sha256": "938a47c7db2d7fcf31849cc970bb6f6fd8a10b05a59378ee3905d6f00493d508"
  },
  "transport_capsule_tar": {
    "bytes": 4733,
    "path": "transport_capsule_epoch_1_signed.tar.gz",
    "sha256": "c4d741a108a93a6eb66a30e36bdbe971530dc5a82966ca95bbb98749621393e7"
  }
}

with open('host_a_build_report.json', 'w') as f:
    json.dump(host_a_report, f, indent=2)
print('host_a_build_report.json written')

## Step 5: Verify file integrity before execution

In [ ]:
report = json.load(open('host_a_build_report.json'))

if os.path.exists('run_on_host_b.py'):
    with open('run_on_host_b.py', 'rb') as f:
        runner_hash = hashlib.sha256(f.read()).hexdigest()
    expected = report['transport_bundle']['sha256']
    match = runner_hash == expected
    print(f'Runner SHA-256:    {runner_hash}')
    print(f'Expected (Host A): {expected}')
    print(f'Match: {"YES" if match else "NO - DO NOT PROCEED"}')
else:
    print('run_on_host_b.py not found')

print()

if os.path.exists('transport_capsule_epoch_1_signed.tar.gz'):
    with open('transport_capsule_epoch_1_signed.tar.gz', 'rb') as f:
        cap_hash = hashlib.sha256(f.read()).hexdigest()
    expected_cap = report['transport_capsule_tar']['sha256']
    match_cap = cap_hash == expected_cap
    print(f'Capsule SHA-256:   {cap_hash}')
    print(f'Expected (Host A): {expected_cap}')
    print(f'Match: {"YES" if match_cap else "NO - DO NOT PROCEED"}')
else:
    print('transport_capsule_epoch_1_signed.tar.gz not found')

## Step 6: Run Host B

This executes the full continuation chain on Linux (Colab), verifying:
1. Runner self-hash
2. External bundle hash against Host A report
3. Owner Ed25519 signature
4. Workspace restoration
5. Deterministic task continuation
6. Successor capsule sealing
7. Host B Ed25519 report signing
8. Evidence packet generation

In [ ]:
import subprocess

RUNNER_SHA = report['transport_bundle']['sha256']
OWNER_PUB = report['capsule_epoch_1']['owner_public_key']

cmd = [
    'python3', 'run_on_host_b.py',
    '--out', '/content/hdar-host-b-proof',
    '--host-label', 'colab-linux-independent-host-b',
    '--host-a-report', 'host_a_build_report.json',
    '--verify-runner-hash', RUNNER_SHA,
    '--owner-public-key', OWNER_PUB,
    '--operator-identity', 'colab-independent-run'
]

print('Running Host B on Colab Linux...')
print(f'Command: {" ".join(cmd)}')
print()

result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:')
print(result.stdout[:5000])
if result.stderr:
    print('STDERR:')
    print(result.stderr[:3000])
print(f'\nExit code: {result.returncode}')

## Step 7: Examine the Host B report

In [ ]:
report_path = '/content/hdar-host-b-proof/host_b_report.json'
if os.path.exists(report_path):
    r = json.load(open(report_path))
    print('=== Host B Report Summary ===')
    print(f'Host B platform:  {r["host_b_platform"]}')
    print(f'Host A platform:   {r["host_a_report_verification"]["host_a_platform"]}')
    print(f'Platforms differ:  {r["host_a_report_verification"]["platforms_differ"]}')
    print(f'Owner signature:   {r["input_capsule"]["owner_signature_verified"]["ok"]}')
    print(f'Bundle verified:   {r["host_a_report_verification"]["external_bundle_hash_match"]}')
    print(f'Restore exact:     {r["restore"]["exact"]}')
    print(f'Task continuation: {r["task_continuation"]["ok"]}')
    print(f'Lineage advanced:  {r["lineage_advanced"]}')
    print(f'Successor epoch:   {r["successor_capsule"]["epoch"]}')
    print(f'Host B signature:  {r["host_b_keypair"]["algorithm"]}')
    print(f'Host B public key: {r["host_b_public_key"]}')
    print()
    print('=== Console Transcript ===')
    for line in r['console_transcript']:
        print(f'  {line}')
else:
    print('host_b_report.json not found - check for errors above')

## Step 8: Third-party verification

Independently verify the full chain.

In [ ]:
import json, hashlib, os
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
from cryptography.exceptions import InvalidSignature

proof_dir = '/content/hdar-host-b-proof'
report = json.load(open(os.path.join(proof_dir, 'host_b_report.json')))

def canonical_json(data):
    return json.dumps(data, sort_keys=True, separators=(',', ':'), ensure_ascii=True).encode()

checks = []

# 1. Owner signature on E1
e1m = json.load(open(os.path.join(proof_dir, 'capsule_epoch_1', 'manifest.json')))
pub_hex = report['input_capsule']['owner_signature_verified']['owner_public_key']
sig_hex = e1m.get('owner_signature', '')
if sig_hex:
    try:
        pk = Ed25519PublicKey.from_public_bytes(bytes.fromhex(pub_hex))
        content = {k: v for k, v in e1m.items() if k not in ('owner_signature', 'owner_public_key', 'manifest_hash')}
        pk.verify(bytes.fromhex(sig_hex), canonical_json(content))
        checks.append(('owner_signature', True, 'Owner Ed25519 signature verified'))
    except InvalidSignature:
        checks.append(('owner_signature', False, 'Owner signature INVALID'))
else:
    checks.append(('owner_signature', False, 'No owner signature'))

# 2. Platforms differ
ha = report['host_a_report_verification']['host_a_platform']
hb = report['host_b_platform']
checks.append(('platforms_differ', ha != hb, f'Host A: {ha} | Host B: {hb}'))

# 3. Task continuation
checks.append(('task_continuation', report['task_continuation']['ok'], f"Result: {report['task_continuation'].get('computed_result', 'N/A')}"))

# 4. Lineage
e2m = json.load(open(os.path.join(proof_dir, 'capsule_epoch_2', 'manifest.json')))
lin = e2m.get('parent_manifest_hash') == e1m.get('manifest_hash') and e2m.get('epoch') == e1m.get('epoch', 0) + 1
checks.append(('lineage', lin, f'E1 epoch={e1m.get("epoch")}, E2 epoch={e2m.get("epoch")}'))

# 5. State advancement
e1r = e1m.get('workspace_manifest', {}).get('root_hash', '')
e2r = e2m.get('workspace_manifest', {}).get('root_hash', '')
checks.append(('state_advanced', e1r != e2r, 'Workspace root hash changed'))

# 6. Host B signature
try:
    hbpk = Ed25519PublicKey.from_public_bytes(bytes.fromhex(report['host_b_public_key']))
    r4s = {k: v for k, v in report.items() if k != 'host_b_signature'}
    hbpk.verify(bytes.fromhex(report['host_b_signature']), canonical_json(r4s))
    checks.append(('host_b_signature', True, 'Host B Ed25519 signature verified'))
except InvalidSignature:
    checks.append(('host_b_signature', False, 'Host B signature INVALID'))
except Exception as e:
    checks.append(('host_b_signature', False, f'Error: {e}'))

# 7. Evidence packet signature
try:
    ep = json.load(open(os.path.join(proof_dir, 'host_b_evidence_packet.json')))
    eppk = Ed25519PublicKey.from_public_bytes(bytes.fromhex(ep['evidence_packet_public_key']))
    ep4s = {k: v for k, v in ep.items() if k not in ('evidence_packet_signature',)}
    eppk.verify(bytes.fromhex(ep['evidence_packet_signature']), canonical_json(ep4s))
    checks.append(('evidence_packet_signature', True, 'Evidence packet Ed25519 signature verified'))
except Exception as e:
    checks.append(('evidence_packet_signature', False, f'Error: {e}'))

# Print results
print('=== Third-Party Verifier Results ===')
print()
passed = sum(1 for _, ok, _ in checks if ok)
total = len(checks)
for name, ok, reason in checks:
    print(f'  [{"PASS" if ok else "FAIL"}] {name}: {reason}')
print()
print(f'  {passed}/{total} checks passed')
if passed == total:
    print('  ALL CHECKS PASSED - independent Host B execution verified on Linux')
else:
    print(f'  {total - passed} checks FAILED')

## Step 9: Package results for download

In [ ]:
import tarfile, os

proof_dir = '/content/hdar-host-b-proof'
if os.path.exists(proof_dir):
    print('Output files:')
    for f in sorted(os.listdir(proof_dir)):
        fpath = os.path.join(proof_dir, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath)
            with open(fpath, 'rb') as fh:
                h = hashlib.sha256(fh.read()).hexdigest()
            print(f'  {f}: {size} bytes, SHA-256={h}')
        elif os.path.isdir(fpath):
            print(f'  {f}/ (directory)')

    tar_path = '/content/hdar_host_b_colab_results.tar.gz'
    with tarfile.open(tar_path, 'w:gz') as tf:
        tf.add(proof_dir, arcname='hdar-host-b-proof')
    print(f'\nResults packaged: {tar_path}')
    print('Download this file from the Colab file browser.')
else:
    print('Proof directory not found')

## What this proves

If all checks passed, this notebook demonstrated:

1. **Cross-platform restoration**: Capsule created on macOS restored exactly on Linux (Colab)
2. **Owner authorization**: Ed25519 owner signature verified on a different OS
3. **External hash verification**: Bundle hash cross-checked against Host A build report
4. **Deterministic continuation**: Task produced expected result on independent hardware
5. **Successor capsule**: Cryptographically linked epoch-2 capsule created on Linux
6. **Host B attestation**: Host B signed report with its own Ed25519 key
7. **Independent verification**: Third-party verifier confirmed the full chain
8. **Genuinely independent host**: `platforms_differ=true` because macOS != Linux, different hardware, different network, different administrative domain

This moves the claim boundary from 'local simulation' to 'independently reproduced cross-platform continuation.'